# Step 3: Feature Engineering & Data Preparation
## AI-Assisted Health Risk Prediction for Life Insurance Underwriting

This notebook turns the merged raw dataset into model-ready train/test sets.
It is **driven by the decisions saved in Step 2** (`data/processed/eda_findings.json`),
so EDA actually controls the pipeline instead of being redone by hand.

**Pipeline:**
1. Load data + EDA findings
2. Select features (drop leakage + high-missing columns, per findings)
3. Clean NHANES sentinel codes (7/9, 77/99, 7777/9999 -> NaN)
4. Create clinically meaningful derived features
5. Stratified train/test split (BEFORE fitting any transformer -> no leakage)
6. Impute missing values (fit on train only)
7. Log-transform skewed continuous features (continuous-only subset)
8. One-hot encode categoricals, standardize numerics
9. Save prepared datasets + fitted preprocessor + metadata

## 3.1 Setup & Load Inputs

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
print('Imports OK')

Imports OK


In [2]:
# Robust path resolution: works whether the notebook runs from the project
# root or from the Code/ subfolder. We search upward for the 'data' folder.
def find_data_dir():
    for base in [Path('.'), Path('..'), Path('..') / '..']:
        candidate = base / 'data' / 'processed'
        if candidate.exists():
            return (base / 'data').resolve()
    raise FileNotFoundError('Could not locate the data/ folder. Run notebooks 01 and 02 first.')

DATA_DIR = find_data_dir()
PROCESSED_DIR = DATA_DIR / 'processed'
print(f'Using DATA_DIR: {DATA_DIR}')

# Load merged dataset from Step 1
parquet_path = PROCESSED_DIR / 'nhanes_mortality_merged.parquet'
csv_path = PROCESSED_DIR / 'nhanes_mortality_merged.csv'
df = pd.read_parquet(parquet_path) if parquet_path.exists() else pd.read_csv(csv_path)
print(f'Loaded dataset: {df.shape}')

# Load EDA findings from Step 2
with open(PROCESSED_DIR / 'eda_findings.json') as f:
    findings = json.load(f)
print('Loaded EDA findings. Keys:', list(findings.keys()))

Using DATA_DIR: <PROJECT_ROOT>\data
Loaded dataset: (18578, 60)
Loaded EDA findings. Keys: ['class_balance', 'high_missing_threshold_pct', 'drop_high_missing', 'missing_pct', 'skew_threshold', 'skewness', 'log_transform_candidates', 'columns', 'abs_corr_with_target']


## 3.2 Feature Selection (driven by EDA findings)

We start from the column typing saved in Step 2 and remove:
- **Leakage columns** (only known after death, e.g. `UCOD_LEADING`, `PERMTH_*`)
- **High-missing columns** (>40% missing, e.g. `SMQ040`, `DIQ070`)

We also drop `LBXSGL` (glucose) because EDA showed it is 0.77 correlated with
`LBXGH` (HbA1c) -- redundant, and HbA1c is the stronger clinical marker.

In [3]:
TARGET = findings['columns']['target']       # MORTSTAT
ID = findings['columns']['id']                # SEQN
leakage = set(findings['columns']['leakage_exclude'])
high_missing = set(findings['drop_high_missing'])

# Manual redundancy drop identified in EDA correlation matrix
redundant = {'LBXSGL'}   # glucose ~0.77 corr with HbA1c (LBXGH)

exclude = leakage | high_missing | redundant | {TARGET, ID}

categorical = [c for c in findings['columns']['categorical']
               if c in df.columns and c not in exclude]
continuous = [c for c in findings['columns']['continuous']
              if c in df.columns and c not in exclude]

print(f'Excluded ({len(exclude)}): {sorted(exclude)}')
print(f'\nCategorical features ({len(categorical)}): {categorical}')
print(f'\nContinuous features ({len(continuous)}): {continuous}')

Excluded (13): ['ALQ130', 'DIQ070', 'ELIGSTAT', 'LBXSGL', 'MORTSTAT', 'MORT_DIABETES', 'MORT_HYPERTEN', 'PERMTH_EXM', 'PERMTH_INT', 'SEQN', 'SMD650', 'SMQ040', 'UCOD_LEADING']

Categorical features (20): ['GENDER', 'RACE_ETHNICITY', 'EDUCATION', 'MARITAL_STATUS', 'CYCLE', 'SMQ020', 'DIQ010', 'DIQ050', 'MCQ160B', 'MCQ160C', 'MCQ160D', 'MCQ160E', 'MCQ160F', 'MCQ220', 'PAQ605', 'PAQ620', 'PAQ635', 'PAQ650', 'PAQ665', 'HIQ011']

Continuous features (27): ['AGE', 'POVERTY_INCOME_RATIO', 'BMXBMI', 'BMXWAIST', 'BMXHT', 'BMXWT', 'BPXSY1', 'BPXDI1', 'BPXSY2', 'BPXDI2', 'LBXTC', 'LBDHDD', 'LBXGH', 'LBXWBCSI', 'LBXRBCSI', 'LBXHGB', 'LBXPLTSI', 'LBXSCR', 'LBXSUA', 'LBXSAL', 'LBXSASSI', 'LBXSATSI', 'LBXSGB', 'LBXSTR', 'URXUMA', 'URXUCR', 'URDACT']


## 3.3 Clean NHANES Sentinel Codes

NHANES questionnaire variables encode non-answers as `7`/`77`/`7777` (Refused)
and `9`/`99`/`9999` (Don't know). If left in, these fake numbers corrupt both
imputation and modeling. We convert them to `NaN` so they are imputed properly.

We only apply this to categorical questionnaire codes (small-valued yes/no items),
not to continuous lab/exam values where 7/9 are legitimate measurements.

In [4]:
df_clean = df.copy()

# Questionnaire yes/no style items: 1=Yes, 2=No, 7=Refused, 9=Don't know
# Single-digit sentinels (7/9).
sentinel_single = ['SMQ020', 'DIQ010', 'DIQ050',
                   'MCQ160B', 'MCQ160C', 'MCQ160D', 'MCQ160E', 'MCQ160F', 'MCQ220',
                   'PAQ605', 'PAQ620', 'PAQ635', 'PAQ650', 'PAQ665', 'HIQ011']

# Multi-category items use two-digit sentinels: 77=Refused, 99=Don't know
# (e.g., EDUCATION 1-5 valid + 7/9; MARITAL_STATUS 1-6 valid + 77/99)
sentinel_multi = ['EDUCATION', 'MARITAL_STATUS']

cleaned_counts = {}
for col in sentinel_single:
    if col in df_clean.columns:
        mask = df_clean[col].isin([7, 9])
        if mask.sum():
            df_clean.loc[mask, col] = np.nan
            cleaned_counts[col] = int(mask.sum())

for col in sentinel_multi:
    if col in df_clean.columns:
        # cover both single (7/9) and double (77/99) digit refusal codes
        mask = df_clean[col].isin([7, 9, 77, 99])
        if mask.sum():
            df_clean.loc[mask, col] = np.nan
            cleaned_counts[col] = int(mask.sum())

print('Sentinel values converted to NaN:')
print(cleaned_counts if cleaned_counts else '  (none found)')

Sentinel values converted to NaN:
{'SMQ020': 11, 'DIQ010': 14, 'MCQ160B': 57, 'MCQ160C': 76, 'MCQ160D': 57, 'MCQ160E': 34, 'MCQ160F': 29, 'MCQ220': 22, 'PAQ605': 2, 'PAQ620': 3, 'PAQ635': 1, 'PAQ650': 2, 'PAQ665': 2, 'HIQ011': 24, 'EDUCATION': 26, 'MARITAL_STATUS': 15}


## 3.4 Create Clinically Meaningful Derived Features

Underwriting uses interpretable clinical thresholds. We engineer a few features
that combine raw measurements into recognized risk indicators:
- **Mean blood pressure** from the two readings (more stable than a single reading)
- **Pulse pressure** (systolic - diastolic): a known cardiovascular risk marker
- **BMI category** (WHO thresholds)
- **Hypertension flag** (>=130/80, current ACC/AHA guideline)

In [5]:
# Mean BP from the two readings (fall back to whichever exists)
if {'BPXSY1', 'BPXSY2'}.issubset(df_clean.columns):
    df_clean['BP_SYS_MEAN'] = df_clean[['BPXSY1', 'BPXSY2']].mean(axis=1)
if {'BPXDI1', 'BPXDI2'}.issubset(df_clean.columns):
    df_clean['BP_DIA_MEAN'] = df_clean[['BPXDI1', 'BPXDI2']].mean(axis=1)

# Pulse pressure
if {'BP_SYS_MEAN', 'BP_DIA_MEAN'}.issubset(df_clean.columns):
    df_clean['PULSE_PRESSURE'] = df_clean['BP_SYS_MEAN'] - df_clean['BP_DIA_MEAN']

# Hypertension flag (ACC/AHA 2017: >=130 systolic OR >=80 diastolic)
if {'BP_SYS_MEAN', 'BP_DIA_MEAN'}.issubset(df_clean.columns):
    df_clean['HYPERTENSION_FLAG'] = (
        (df_clean['BP_SYS_MEAN'] >= 130) | (df_clean['BP_DIA_MEAN'] >= 80)
    ).astype(float)

# BMI category (numeric ordinal: 0=under,1=normal,2=over,3=obese)
if 'BMXBMI' in df_clean.columns:
    df_clean['BMI_CATEGORY'] = pd.cut(
        df_clean['BMXBMI'], bins=[0, 18.5, 25, 30, np.inf],
        labels=[0, 1, 2, 3]).astype('float')

new_continuous = [c for c in ['BP_SYS_MEAN', 'BP_DIA_MEAN', 'PULSE_PRESSURE'] if c in df_clean.columns]
new_categorical = [c for c in ['HYPERTENSION_FLAG', 'BMI_CATEGORY'] if c in df_clean.columns]

# Replace the raw double-reading BP columns with the engineered means
for raw in ['BPXSY1', 'BPXSY2', 'BPXDI1', 'BPXDI2']:
    if raw in continuous:
        continuous.remove(raw)

continuous = continuous + new_continuous
categorical = categorical + new_categorical
print(f'Derived continuous: {new_continuous}')
print(f'Derived categorical: {new_categorical}')
print(f'\nFinal continuous ({len(continuous)}): {continuous}')
print(f'Final categorical ({len(categorical)}): {categorical}')

Derived continuous: ['BP_SYS_MEAN', 'BP_DIA_MEAN', 'PULSE_PRESSURE']
Derived categorical: ['HYPERTENSION_FLAG', 'BMI_CATEGORY']

Final continuous (26): ['AGE', 'POVERTY_INCOME_RATIO', 'BMXBMI', 'BMXWAIST', 'BMXHT', 'BMXWT', 'LBXTC', 'LBDHDD', 'LBXGH', 'LBXWBCSI', 'LBXRBCSI', 'LBXHGB', 'LBXPLTSI', 'LBXSCR', 'LBXSUA', 'LBXSAL', 'LBXSASSI', 'LBXSATSI', 'LBXSGB', 'LBXSTR', 'URXUMA', 'URXUCR', 'URDACT', 'BP_SYS_MEAN', 'BP_DIA_MEAN', 'PULSE_PRESSURE']
Final categorical (22): ['GENDER', 'RACE_ETHNICITY', 'EDUCATION', 'MARITAL_STATUS', 'CYCLE', 'SMQ020', 'DIQ010', 'DIQ050', 'MCQ160B', 'MCQ160C', 'MCQ160D', 'MCQ160E', 'MCQ160F', 'MCQ220', 'PAQ605', 'PAQ620', 'PAQ635', 'PAQ650', 'PAQ665', 'HIQ011', 'HYPERTENSION_FLAG', 'BMI_CATEGORY']


## 3.5 Assemble X / y and Stratified Train/Test Split

We drop rows with a missing target, then split **before** fitting any imputer or
scaler. Fitting transformers only on the training set prevents data leakage from
test into train. The split is stratified on `MORTSTAT` to preserve the ~14%
mortality rate in both sets.

In [7]:
feature_cols = continuous + categorical

# Keep SEQN alongside features so the demo can label/select patients
model_df = df_clean[[ID] + feature_cols + [TARGET]].copy()
model_df = model_df[model_df[TARGET].notna()]      # need a label
model_df[TARGET] = model_df[TARGET].astype(int)

X = model_df[feature_cols]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

print(f'Train: {X_train.shape},  Test: {X_test.shape}')
print(f'Train mortality rate: {y_train.mean():.4f}')
print(f'Test  mortality rate: {y_test.mean():.4f}')

# --- Save RAW (human-readable, pre-log-transform) feature frames for the demo UI ---
# The demo re-applies log1p + the saved preprocessor, so it needs raw values here.
RAW_DIR = DATA_DIR / 'model_ready'
RAW_DIR.mkdir(parents=True, exist_ok=True)
X_test_raw = X_test.copy()
X_test_raw[ID] = model_df.loc[X_test.index, ID].values
X_test_raw[TARGET] = y_test.values
X_test_raw.to_parquet(RAW_DIR / 'X_test_raw.parquet')
# Training-set medians/modes as defaults for the manual-entry form
raw_defaults = {}
for c in continuous:
    raw_defaults[c] = float(X_train[c].median())
for c in categorical:
    mode = X_train[c].mode(dropna=True)
    if len(mode):
        val = mode.iloc[0]
        # keep strings (e.g., CYCLE='2009-2010') as-is; cast numpy numbers to float
        raw_defaults[c] = val if isinstance(val, str) else float(val)
    else:
        raw_defaults[c] = 0.0
import json as _json
with open(RAW_DIR / 'raw_feature_defaults.json', 'w') as _f:
    _json.dump(raw_defaults, _f, indent=2)
print(f'Saved X_test_raw ({X_test_raw.shape}) and raw_feature_defaults.json for the demo.')

Train: (14862, 48),  Test: (3716, 48)
Train mortality rate: 0.1408
Test  mortality rate: 0.1407
Saved X_test_raw ((3716, 50)) and raw_feature_defaults.json for the demo.


## 3.6 Log-Transform Skewed Continuous Features

Step 2 flagged skewed columns, but that list was computed over *all* numeric
columns and wrongly included categorical codes. Here we take the **intersection**
with our true continuous features, then apply `log1p` (handles zeros safely).
This mainly helps the GLM baseline; it is harmless for tree models.

The transform is applied in-place (replaces the column) because the log and raw
versions carry the same information -- keeping both would be redundant.

In [8]:
skew_candidates = set(findings['log_transform_candidates'])
log_cols = [c for c in continuous if c in skew_candidates]
# Only log-transform strictly non-negative columns (log1p needs x > -1)
log_cols = [c for c in log_cols if (X_train[c].min(skipna=True) >= 0)]
print(f'Log-transforming {len(log_cols)} continuous features:')
print(log_cols)

for c in log_cols:
    X_train[c] = np.log1p(X_train[c])
    X_test[c] = np.log1p(X_test[c])
print('\nApplied log1p (train + test).')

Log-transforming 12 continuous features:
['BMXBMI', 'BMXWT', 'LBDHDD', 'LBXGH', 'LBXWBCSI', 'LBXSCR', 'LBXSASSI', 'LBXSATSI', 'LBXSTR', 'URXUMA', 'URXUCR', 'URDACT']

Applied log1p (train + test).


## 3.7 Impute, Scale & Encode via a ColumnTransformer

A single `ColumnTransformer` handles both column types:
- **Continuous:** median imputation -> standardization (z-score)
- **Categorical:** most-frequent imputation -> one-hot encoding

It is `fit` on the **training set only**, then used to transform both sets.

In [10]:
continuous_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', continuous_pipe, continuous),
    ('cat', categorical_pipe, categorical),
])

# Fit on train, transform both
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Recover feature names after one-hot
feature_names = preprocessor.get_feature_names_out()
X_train_proc = pd.DataFrame(X_train_proc, columns=feature_names, index=X_train.index)
X_test_proc = pd.DataFrame(X_test_proc, columns=feature_names, index=X_test.index)

#print (feature_names)

print(f'Processed train: {X_train_proc.shape}')
print(f'Processed test:  {X_test_proc.shape}')
print(f'Total features after encoding: {len(feature_names)}')

Processed train: (14862, 84)
Processed test:  (3716, 84)
Total features after encoding: 84


## 3.8 Save Prepared Data, Preprocessor & Metadata

Everything Steps 4-7 need is persisted:
- Processed `X`/`y` for train and test (parquet)
- The fitted `preprocessor` (joblib) so the LLM/inference step can transform new rows
- A metadata JSON (feature names, class weight, transform lists)

In [11]:
MODEL_DIR = DATA_DIR / 'model_ready'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save processed matrices
X_train_proc.to_parquet(MODEL_DIR / 'X_train.parquet')
X_test_proc.to_parquet(MODEL_DIR / 'X_test.parquet')
y_train.to_frame('MORTSTAT').to_parquet(MODEL_DIR / 'y_train.parquet')
y_test.to_frame('MORTSTAT').to_parquet(MODEL_DIR / 'y_test.parquet')

# Save fitted preprocessor for inference on new data
joblib.dump(preprocessor, MODEL_DIR / 'preprocessor.joblib')

# Save metadata for downstream notebooks
prep_meta = {
    'target': TARGET,
    'continuous_features': continuous,
    'categorical_features': categorical,
    'log_transformed': log_cols,
    'encoded_feature_names': feature_names.tolist(),
    'n_features_out': int(len(feature_names)),
    'scale_pos_weight': findings['class_balance']['scale_pos_weight'],
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'random_state': RANDOM_STATE,
}
with open(MODEL_DIR / 'preprocessing_metadata.json', 'w') as f:
    json.dump(prep_meta, f, indent=2)

print('Saved to', MODEL_DIR)
for p in sorted(MODEL_DIR.iterdir()):
    print(' -', p.name)

Saved to <PROJECT_ROOT>\data\model_ready
 - preprocessing_metadata.json
 - preprocessor.joblib
 - raw_feature_defaults.json
 - X_test.parquet
 - X_test_raw.parquet
 - X_train.parquet
 - y_test.parquet
 - y_train.parquet


## Next Steps

Model-ready data is in `data/model_ready/`. Proceed to **Notebook 04: GLM Baseline**.

Downstream notebooks should:
1. Load `X_train.parquet` / `y_train.parquet` etc.
2. Read `scale_pos_weight` from `preprocessing_metadata.json` for class imbalance
3. Use the saved `preprocessor.joblib` to transform any new/raw records